# Food segmentation v3: baseline / aug / aug + cơm gộp, 3 seed, có tập TEST khoá

Dataset `food_dataset_v3`: train 162 / val 49 / test 36, chia theo buổi chụp. Val giữ nguyên như v2.

| Cấu hình | Thay đổi so với dòng trên |
|---|---|
| `baseline` | yolo26s-seg, tham số mặc định |
| `aug` | + xoay 180°, lật dọc, copy-paste mask (cải tiến #1) |
| `rice1` | + mỗi đĩa chỉ có **một** polygon cơm (cải tiến #2) |

**So sánh không dựa trên mAP của `rice`**, vì `rice1` dùng đáp án cơm khác hai cấu hình kia.
Các chỉ số dùng để quyết định không phụ thuộc cách gán nhãn cơm:
- `rice_iou`: IoU giữa toàn bộ vùng cơm dự đoán và toàn bộ vùng cơm thật, trung bình theo ảnh
- `rice_area_err`: |diện tích dự đoán − thật| / thật. Đây chính là thứ đi vào phép tính gram
- `mIoU_all`: IoU từng lớp trên cả tập test, lấy trung bình
- `mask_AP_excl_rice`: AP mask trung bình của các lớp **trừ cơm**. Đáp án các lớp này giống hệt nhau ở mọi cấu hình

Chạy theo thứ tự **seed trước, cấu hình sau**. Nếu notebook chết giữa chừng thì những seed đã chạy xong vẫn đủ bộ để so.

In [ ]:
!pip install -q ultralytics
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
import glob, yaml, pathlib, shutil, os, collections, json
import numpy as np, pandas as pd, cv2

cands = [p for p in glob.glob('/kaggle/input/**/data.yaml', recursive=True)
         if 'test' in yaml.safe_load(open(p))]
assert cands, 'Chua add food-dataset-v3'
SRC = str(pathlib.Path(cands[0]).parent)
cfg = yaml.safe_load(open(cands[0]))
names = [cfg['names'][i] for i in sorted(cfg['names'])]
RICE = names.index('rice')

def write_yaml(root, path):
    c = dict(cfg); c['path'] = root
    yaml.safe_dump(c, open(path, 'w'), sort_keys=False, allow_unicode=True)
    return path

DATA = write_yaml(SRC, '/kaggle/working/data_v3.yaml')

# Bản cơm gộp: CHÉP ẢNH THẬT, không symlink. Ultralytics gọi .resolve() lên đường dẫn,
# symlink sẽ trỏ ngược về /kaggle/input và nó lặng lẽ đọc nhãn gốc.
R1 = '/kaggle/working/v3_rice1'
if not os.path.exists(R1):
    shutil.copytree(f'{SRC}/images', f'{R1}/images')
    shutil.copytree(f'{SRC}/labels_rice1', f'{R1}/labels')
DATA_R1 = write_yaml(R1, '/kaggle/working/data_rice1.yaml')
for s in ('train', 'val', 'test'):
    n = len(glob.glob(f'{SRC}/images/{s}/*'))
    r = sum(1 for p in glob.glob(f'{R1}/labels/{s}/*.txt') for l in open(p) if l.split()[:1] == [str(RICE)])
    print(f'{s}: {n} anh | polygon rice trong ban gop: {r}')

## Tham số chung + cách chấm điểm

In [ ]:
from ultralytics import YOLO

COMMON = dict(epochs=150, patience=50, imgsz=640, batch=16, device=0, workers=2,
              deterministic=True, project='/kaggle/working/runs', exist_ok=True, plots=False)
AUG = dict(degrees=180, flipud=0.5, fliplr=0.5, scale=0.6, copy_paste=0.3, cos_lr=True, close_mosaic=25)
CONFIGS = {'baseline': (DATA, {}), 'aug': (DATA, AUG), 'rice1': (DATA_R1, AUG)}
SEEDS = [0, 1, 2]
CONF = 0.25   # chốt trước, dùng chung cho mọi cấu hình

TEST_IMGS = sorted(glob.glob(f'{SRC}/images/test/*.jpg'))

def gt_masks(img_path, shape):
    # Đáp án gốc (cơm nhiều mảnh) -> mask hợp theo lớp. Dùng chung cho MỌI cấu hình.
    H, W = shape
    out = {}
    lp = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    for l in open(lp):
        r = l.split()
        if not r: continue
        c = int(r[0]); xy = (np.array(r[1:], float).reshape(-1, 2) * [W, H]).round().astype(np.int32)
        m = out.setdefault(c, np.zeros((H, W), np.uint8)); cv2.fillPoly(m, [xy], 1)
    return out

def free_metrics(w):
    model = YOLO(w)
    inter, union = collections.Counter(), collections.Counter()
    rice_iou, rice_err = [], []
    for p in TEST_IMGS:
        r = model.predict(p, conf=CONF, retina_masks=True, verbose=False)[0]
        H, W = r.orig_shape
        pred = {}
        if r.masks is not None:
            for m, c in zip(r.masks.data.cpu().numpy() > 0.5, r.boxes.cls.cpu().numpy().astype(int)):
                pred[c] = pred.get(c, np.zeros((H, W), bool)) | m
        gt = gt_masks(p, (H, W))
        for c in set(gt) | set(pred):
            g = gt.get(c, np.zeros((H, W), np.uint8)).astype(bool); q = pred.get(c, np.zeros((H, W), bool))
            inter[c] += (g & q).sum(); union[c] += (g | q).sum()
        if RICE in gt:
            g = gt[RICE].astype(bool); q = pred.get(RICE, np.zeros((H, W), bool))
            rice_iou.append((g & q).sum() / max((g | q).sum(), 1))
            rice_err.append(abs(int(q.sum()) - int(g.sum())) / g.sum())
    miou = np.mean([inter[c] / union[c] for c in union if union[c]])
    return dict(rice_iou=np.mean(rice_iou), rice_area_err=np.mean(rice_err), mIoU_all=miou)

def evaluate(w, data, tag):
    out = {}
    for split in ('val', 'test'):
        v = YOLO(w).val(data=data, split=split, imgsz=640, plots=False, verbose=False)
        out[f'{split}_mask_mAP50_95'] = v.seg.map; out[f'{split}_box_mAP50_95'] = v.box.map
        if split == 'test':
            ap = dict(zip(v.ap_class_index, v.seg.ap))
            out['test_mask_AP_excl_rice'] = np.mean([a for c, a in ap.items() if c != RICE])
            per = pd.DataFrame({'lop': [names[c] for c in ap], tag: list(ap.values())})
    out.update(free_metrics(w))
    return out, per

## Train 3 cấu hình × 3 seed. Mỗi lượt xong là ghi kết quả ngay

In [ ]:
rows, per_all = [], None
if os.path.exists('/kaggle/working/summary.csv'):
    rows = pd.read_csv('/kaggle/working/summary.csv').to_dict('records')
done = {(r['cfg'], r['seed']) for r in rows}

for sd in SEEDS:
    for name, (data, aug) in CONFIGS.items():
        if (name, sd) in done: continue
        tag = f'{name}_s{sd}'
        w = f'/kaggle/working/runs/{tag}/weights/best.pt'
        try:
            YOLO('yolo26s-seg.pt').train(data=data, name=tag, seed=sd, **COMMON, **aug)
            # lưu model TRƯỚC khi chấm: chấm lỗi thì chấm lại được, không phải train lại
            shutil.copy2(w, f'/kaggle/working/{tag}_best.pt')
            shutil.copy2(f'/kaggle/working/runs/{tag}/results.csv', f'/kaggle/working/{tag}_results.csv')
        except Exception as e:
            print('LOI TRAIN', tag, repr(e)); rows.append(dict(cfg=name, seed=sd, error='train: ' + repr(e)[:200]))
            pd.DataFrame(rows).to_csv('/kaggle/working/summary.csv', index=False); continue
        try:
            m, per = evaluate(w, data, tag)
            rows.append(dict(cfg=name, seed=sd, **m))
            per_all = per if per_all is None else per_all.merge(per, on='lop', how='outer')
        except Exception as e:
            print('LOI CHAM', tag, repr(e)); rows.append(dict(cfg=name, seed=sd, error='eval: ' + repr(e)[:200]))
        pd.DataFrame(rows).to_csv('/kaggle/working/summary.csv', index=False)
        if per_all is not None: per_all.to_csv('/kaggle/working/per_class_test_ap.csv', index=False)
        print(pd.DataFrame(rows).round(3).to_string())

## Kết quả: trung bình ± độ lệch chuẩn qua 3 seed

In [ ]:
S = pd.DataFrame(rows)
cols = [c for c in S.columns if c not in ('cfg', 'seed', 'error')]
agg = S.groupby('cfg')[cols].agg(['mean', 'std']).round(3).T
agg.to_csv('/kaggle/working/summary_mean_std.csv')
agg

In [ ]:
# Chênh lệch theo từng seed (cùng seed so với nhau)
for a, b in (('aug', 'baseline'), ('rice1', 'aug')):
    pa = S[S.cfg == a].set_index('seed')[cols]; pb = S[S.cfg == b].set_index('seed')[cols]
    print(f'--- {a} - {b}'); print((pa - pb).round(3).to_string())